# India Quarterly Results 2019-2026 (NSE/BSE): starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/india_quarterly_results_sample.csv', recursive=True) or ['india_quarterly_results_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df['quarter_end'] = pd.to_datetime(df.quarter_end)
g = df.dropna(subset=['revenue_yoy_pct']).groupby('quarter_end').revenue_yoy_pct.median()
g.plot.bar(figsize=(12, 4), title='Median revenue growth YoY by quarter (sample)')
g.tail(8)

In [ ]:
latest = df[df.quarter_end == df.groupby('quarter_end').size().loc[lambda s: s >= s.median() * .6].index.max()]
latest.nlargest(20, 'revenue_inr')[['company_name', 'sector', 'revenue_inr', 'net_profit_inr', 'revenue_yoy_pct', 'basis']]

In [ ]:
m = df[(df.revenue_inr > 0) & df.net_profit_inr.notna()].assign(margin=lambda d: (d.net_profit_inr / d.revenue_inr).clip(-2, 2))
m.groupby('sector').margin.median().sort_values().plot.barh(figsize=(8, 7), title='Median net margin by sector (sample)')
m.groupby('sector').margin.describe().sort_values('count', ascending=False).head(15)

## Get the full record from the Radar API

The sample keeps the headline lines. The API returns the full quarterly income statement for a company, plus annual results, balance sheet and cash flow. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/financials', headers={'Authorization': f'Bearer {key}'},
                      json={'period_type': 'quarterly', 'statement': 'income', 'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp